In [ ]:
import numpy as npimport tensorflow as tffrom tensorflow.keras import layers, Modelfrom sklearn.metrics import accuracy_score, classification_report

In [ ]:
def residual_tcn_block(x, filters, dilation_rate, dropout=0.2):    h = layers.Conv1D(filters, 3, padding="same", dilation_rate=dilation_rate)(x)    h = layers.BatchNormalization()(h)    h = layers.Activation("relu")(h)    h = layers.Dropout(dropout)(h)    h = layers.Conv1D(filters, 3, padding="same", dilation_rate=dilation_rate)(h)    h = layers.BatchNormalization()(h)    if x.shape[-1] != filters:        x = layers.Conv1D(filters, 1, padding="same")(x)    h = layers.Add()([x, h])    return layers.Activation("relu")(h)

In [ ]:
def build_1d_tcn(input_shape=(999,1), num_classes=17):    inp = layers.Input(shape=input_shape)    x = inp    for d in [1,2,4,8,16,32,64]:        x = residual_tcn_block(x, 64, d)    x = layers.Conv1D(128, 3, padding="same", activation="relu", name="target_conv_layer")(x)    x = layers.GlobalMaxPooling1D()(x)    x = layers.Dropout(0.4)(x)    out = layers.Dense(num_classes, activation="softmax")(x)    return Model(inp, out)tcn_model = build_1d_tcn()tcn_model.compile(    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),    loss="sparse_categorical_crossentropy",    metrics=["accuracy"])tcn_model.summary()

In [ ]:
@tf.functiondef augment_noise_tf(x, y):    snr = tf.random.uniform([], 20.0, 50.0)    xpow = tf.reduce_mean(tf.square(x), axis=[1,2], keepdims=True)    npow = xpow / (10.0 ** (snr / 10.0))    noise = tf.random.normal(tf.shape(x), stddev=tf.sqrt(npow))    return x + noise, y

In [ ]:
# Replace with your actual dataset# X_train_1d, y_train, X_test_1d, y_test must be definedbatch_size = 32train_ds = tf.data.Dataset.from_tensor_slices((X_train_1d, y_train))train_ds = train_ds.shuffle(4096).batch(batch_size).map(    augment_noise_tf, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)val_ds = tf.data.Dataset.from_tensor_slices((X_test_1d, y_test)).batch(batch_size)

In [ ]:
callbacks = [    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3),    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)]history = tcn_model.fit(    train_ds,    validation_data=val_ds,    epochs=60,    callbacks=callbacks)

In [ ]:
preds = np.argmax(tcn_model.predict(X_test_1d), axis=1)print("Accuracy:", accuracy_score(y_test, preds))print(classification_report(y_test, preds))